In [1]:
# Reload both modules
import importlib
import Market_Expectation_Agent
import Stock_Trend_Read_Agent
import Fundamental_Segmentation_Agent 

importlib.reload(Market_Expectation_Agent)
importlib.reload(Stock_Trend_Read_Agent)
importlib.reload(Fundamental_Segmentation_Agent) 

import Revenue_Segmentation_Read_Agent
importlib.reload(Revenue_Segmentation_Read_Agent)


import json
import asyncio
import yfinance as yf
import time
from LLM_Call_Agent import LLMCallAgent
# pip install langchain-openai pydantic
from langchain_openai import ChatOpenAI
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List

import News_Verification
importlib.reload(News_Verification)
# Use the EXISTING LLM_Call_Agent instead of redefining it!
import LLM_Call_Agent
importlib.reload(LLM_Call_Agent)
from LLM_Call_Agent import LLMCallAgent


/Users/xikinki/anaconda3/envs/arviz_env/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3577: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/xikinki/Desktop/Fintegrate_AI_File/Streamlit_APP V2/News_Verification.py:55: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and sho

In [2]:
REDIS_CONFIG = {
    'host': 'redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com',
    'port': 16376,
    'password': 'rl8242B4UItBhFzgHW5APEqZnkYoaEZv'
}

In [3]:
user_question = "Trump is giving more money to stimulate the user market of BTC, and give more revene part"
user_id = "123"
ticker = "MSTR"

## Step 3). Quries to Impaction System

## Manager Agent 

In [4]:
class Manager_Agent_Result(BaseModel):
    Decision_call_market_expectation: int
    Decision_call_revenue_segmentation: int
    Decision_call_macro_analyst: int
    query_for_market_expectation: str      
    query_for_revenue_segmentation: str   
    query_for_macro_analyst: str           


Manager_agent = LLMCallAgent(  
    default_provider="deepseek",
    default_model="deepseek-chat"
)

# ✅ NOW THIS WORKS! Use the new centralized method!
structured_llm = Manager_agent.get_structured_llm(Manager_Agent_Result)

2025-08-25 17:21:58,140 - INFO - 📥 Using OpenAI API key from centralized keys
2025-08-25 17:21:58,155 - INFO - 📥 Using DeepSeek API key from centralized keys
2025-08-25 17:21:58,244 - INFO - ✅ OpenAI client initialized
2025-08-25 17:21:58,265 - INFO - ✅ DeepSeek client initialized
2025-08-25 17:21:58,267 - INFO - 🤖 LLM Call Agent initialized
2025-08-25 17:21:58,270 - INFO -    - Default provider: deepseek
2025-08-25 17:21:58,272 - INFO -    - Default model: deepseek-chat
2025-08-25 17:21:58,274 - INFO -    - OpenAI: Enabled
2025-08-25 17:21:58,275 - INFO -    - DeepSeek: Enabled


/Users/xikinki/anaconda3/envs/arviz_env/lib/python3.10/site-packages/langchain_openai/chat_models/base.py:1896: UserWarning: Received a Pydantic BaseModel V1 schema. This is not supported by method="json_schema". Please use method="function_calling" or specify schema via JSON Schema or Pydantic V2 BaseModel. Overriding to method="function_calling".
  warnings.warn(


In [5]:
# Add this after your structured_llm setup:
def process_manager_query(user_query: str, ticker: str) -> Manager_Agent_Result:
    """
    Process a user query and intelligently route to appropriate agents
    """
    
    # Your intelligent routing prompt
    prompt = f"""
    You are a Manager Agent that analyzes user queries and decides which specialized agents to call.
    
    USER QUERY: "{user_query}"
    TICKER: {ticker}
    
    AVAILABLE AGENTS AND THEIR CAPABILITIES:
    
    1. MARKET EXPECTATION AGENT:
       - Database: Stock trend time intervals, price behavior, micro/macro events, timeline segmentation
       - Best for: Finding similar historical trends based on events, policies, earnings, Micro/Macro events
       - Example queries: "Given tariff cuts, find similar timeline trends with similar macro/micro events"
       - Decision: Call if query involves market trends, price behavior, or historical pattern matching
    
    2. REVENUE SEGMENTATION AGENT:
       - Database: Revenue breakdown (GPU %, Data Center %, etc.), customer segments
       - Best for: Revenue impact analysis of corporate partnerships, service changes, market shifts, Lawsuit, Cost, Policy/ Law that impact its business. 
       - Example queries: "How will Microsoft partnership affect CRWV revenue segments?"
       - Decision: Call if query involves revenue drivers, partnerships, or business model changes
    
    3. MACRO ANALYST AGENT:
       - Database: Macroeconomic indicators, policy changes, economic environment data
       - Best for: Economic factors affecting stock performance, policy impacts
       - Example queries: "How do interest rates affect CRWV?"
       - Decision: Call if query involves economic environment, policies, or macro factors
    
    YOUR TASK:
    1. Analyze the user query to determine which agents are relevant
    2. Generate specific, targeted queries for each relevant agent
    3. Set decision flags (1=call, 0=don't call) for each agent
    4. Ensure queries are specific and actionable for each agent's database
    
    OUTPUT FORMAT:
    - Decision_call_market_expectation: 1 if query involves trends/patterns, 0 otherwise
    - Decision_call_fundamental_segmentation: 1 if query involves revenue/business model, 0 otherwise  
    - Decision_call_macro_analyst: 1 if query involves economic environment, 0 otherwise
    - Generate specific queries only for agents you decide to call (1)
    - For agents you don't call (0), use "N/A" as the query
    """
    
    # Call the structured LLM
    result = structured_llm.invoke(prompt)
    return result



## Manager Agent Calling 

In [6]:
result = process_manager_query(user_question, ticker)

print("🤖 Manager Agent Analysis Results:")
print("=" * 50)
print(f"User Query: {user_question}")
print(f"Ticker: {ticker}")
print("\n📊 Agent Routing Decisions:")
print(f"Market Expectation Agent: {'✅ CALL' if result.Decision_call_market_expectation else '❌ SKIP'}")
print(f"Revenue Segmentation Agent: {'✅ CALL' if result.Decision_call_revenue_segmentation else '❌ SKIP'}")
print(f"Macro Analyst Agent: {'✅ CALL' if result.Decision_call_macro_analyst else '❌ SKIP'}")

print("\n🔍 Generated Queries:")

if result.Decision_call_market_expectation:
    print(f"Market: {result.query_for_market_expectation}")
if result.Decision_call_revenue_segmentation:
    print(f"Revenue: {result.query_for_revenue_segmentation}")
if result.Decision_call_macro_analyst:
    print(f"Macro: {result.query_for_macro_analyst}")

2025-08-25 17:21:58,720 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
🤖 Manager Agent Analysis Results:
User Query: Trump is giving more money to stimulate the user market of BTC, and give more revene part
Ticker: MSTR

📊 Agent Routing Decisions:
Market Expectation Agent: ✅ CALL
Revenue Segmentation Agent: ✅ CALL
Macro Analyst Agent: ✅ CALL

🔍 Generated Queries:
Market: Find historical Bitcoin-related market trends for MSTR when there were government stimulus announcements or monetary policy changes affecting cryptocurrency markets, particularly looking for similar timeline patterns and price behavior
Revenue: Analyze how government stimulus policies and cryptocurrency market conditions impact MSTR's revenue segments and business model, particularly focusing on Bitcoin holdings and related business operations
Macro: Analyze the macroeconomic impact of government stimulus policies on cryptocurrency markets and how this affects MSTR's stock perfo

## Agent_Calling_Form

In [7]:
Decision_call_market_expectation = result.Decision_call_market_expectation
Decision_call_revenue_segmentation = result.Decision_call_revenue_segmentation
Decision_call_macro_analyst = result.Decision_call_macro_analyst

Market_Expectation_Agent_Query = result.query_for_market_expectation
Revenue_Segmentation_Query = result.query_for_revenue_segmentation
Macro_Query = result.query_for_macro_analyst

Decision_List = [Decision_call_market_expectation, Decision_call_revenue_segmentation, Decision_call_macro_analyst]
Agent_List = ["Market_Expectation_Agent", "Revenue_Segmentation_Agent", "Macro_Analyst_Agent"]
Query_List = [Market_Expectation_Agent_Query, Revenue_Segmentation_Query, Macro_Query]

Agents_Calling_Form = {}
for i in range(len(Decision_List)):
    if Decision_List[i] == 1:
        Agents_Calling_Form[Agent_List[i]] = 1
        Agents_Calling_Form[Agent_List[i] + "_Query"] = Query_List[i]

print(Agents_Calling_Form)

{'Market_Expectation_Agent': 1, 'Market_Expectation_Agent_Query': 'Find historical Bitcoin-related market trends for MSTR when there were government stimulus announcements or monetary policy changes affecting cryptocurrency markets, particularly looking for similar timeline patterns and price behavior', 'Revenue_Segmentation_Agent': 1, 'Revenue_Segmentation_Agent_Query': "Analyze how government stimulus policies and cryptocurrency market conditions impact MSTR's revenue segments and business model, particularly focusing on Bitcoin holdings and related business operations", 'Macro_Analyst_Agent': 1, 'Macro_Analyst_Agent_Query': "Analyze the macroeconomic impact of government stimulus policies on cryptocurrency markets and how this affects MSTR's stock performance, including monetary policy changes and regulatory environment"}


## Sub Agent Calling 

In [8]:
# Add this in a NEW cell (Cell 8)
async def call_agents_dynamically():
    """
    Dynamically call agents based on decision flags and collect results
    """
    # Use the variables you already defined in Cell 7
    global Decision_call_market_expectation, Decision_call_revenue_segmentation, Decision_call_macro_analyst
    global Market_Expectation_Agent_Query, Revenue_Segmentation_Query, Macro_Query
    global REDIS_CONFIG, ticker
    
    # Decision and agent mapping
    global Decision_List 
    global Agent_List
    global Query_List
    
    # Initialize results dictionary
    Agents_Results = {}
    
    print("🚀 Starting Dynamic Agent Execution...")
    print("=" * 60)
    
    # Dynamically call agents based on decisions
    for i in range(len(Decision_List)):
        if Decision_List[i] == 1:
            agent_name = Agent_List[i]
            agent_query = Query_List[i]
            
            print(f"✅ Calling {agent_name}...")
            print(f"📝 Query: {agent_query[:100]}...")
            
            try:
                # Call appropriate agent based on name
                if agent_name == "Market_Expectation_Agent":
                    from Market_Expectation_Agent import MarketExpectationAgent
                    agent = MarketExpectationAgent(
                        redis_host=REDIS_CONFIG['host'],
                        redis_port=REDIS_CONFIG['port'],
                        redis_password=REDIS_CONFIG['password']
                    )
                    agent_result = await agent.process_query(agent_query, ticker)
                    Agents_Results[f"{agent_name}_Result"] = agent_result.get('stock_read_result', 'No result')
                    agent.close()
                    
                elif agent_name == "Revenue_Segmentation_Agent":
                    from Fundamental_Segmentation_Agent import FundamentalSegmentationAgent
                    agent = FundamentalSegmentationAgent(
                        redis_host=REDIS_CONFIG['host'],
                        redis_port=REDIS_CONFIG['port'],
                        redis_password=REDIS_CONFIG['password']
                    )
                    agent_result = await agent.process_query(agent_query, ticker)
                    Agents_Results[f"{agent_name}_Result"] = agent_result.get('revenue_analysis', 'No result')
                    agent.close()
                    
                elif agent_name == "Macro_Analyst_Agent":
                    from Macro_Analyst_Agent import MacroAnalystAgent
                    agent = MacroAnalystAgent(user_id="Fintegrate_AI_Test")
                    agent_result = agent.process_macro_query(agent_query)
                    Agents_Results[f"{agent_name}_Result"] = agent_result.get('analysis', 'No result')
                    if hasattr(agent, 'redis_client') and agent.redis_client:
                        agent.redis_client.close()
                
                print(f"✅ {agent_name} completed successfully")
                
            except Exception as e:
                print(f"❌ Error in {agent_name}: {e}")
                Agents_Results[f"{agent_name}_Result"] = f"Error: {str(e)}"
        else:
            print(f"⏭️ Skipping {Agent_List[i]} (decision = 0)")
    
    print("\n" + "=" * 60)
    print("📊 Final Results Summary:")
    print("=" * 60)
    
    # Display results
    for key, value in Agents_Results.items():
        print(f"{key}: {str(value)[:200]}...")
    
    return Agents_Results

In [9]:
# Execute the dynamic agent calling pipeline
print("🚀 Starting Dynamic Agent Pipeline...")
print("=" * 60)

# Run the dynamic agent calling function
final_results = await call_agents_dynamically()

# Create structured output format
agents_result = ""
for key, value in final_results.items():
    agents_result += f"{{{key}: {str(value)[:100]}...}} "

print(f"\n🎯 Final Structured Output:")
print("=" * 60)
print(agents_result)

# Store results for further use
print(f"\n📊 Results Summary:")
print(f"Total agents called: {len(final_results)}")
print(f"Available results: {list(final_results.keys())}")


🚀 Starting Dynamic Agent Pipeline...
🚀 Starting Dynamic Agent Execution...
✅ Calling Market_Expectation_Agent...
📝 Query: Find historical Bitcoin-related market trends for MSTR when there were government stimulus announcem...
2025-08-25 17:22:15,388 - INFO - ✅ Frontend Redis connected: redis-16204.fcrce180.us-east-1-1.ec2.redns.redis-cloud.com:16204
2025-08-25 17:22:15,393 - INFO - ✅ Stock trend Redis connected: redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com:16376
2025-08-25 17:22:15,400 - INFO - Attempting to connect to Redis...
2025-08-25 17:22:15,405 - INFO - Host: redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com
2025-08-25 17:22:15,408 - INFO - Port: 16376
2025-08-25 17:22:15,415 - INFO - Username: default
2025-08-25 17:22:15,422 - INFO - Testing connection with ping command...
2025-08-25 17:22:15,600 - INFO - ✓ Ping successful - Redis server is reachable
2025-08-25 17:22:15,600 - INFO - ✓ Successfully connected to Redis
2025-08-25 17:22:15,602 - INFO - 📥 Using 

In [10]:
print(final_results["Market_Expectation_Agent_Result"])

✅ Using fresh data for MSTR. **SIMILAR TREND MAPPING**: 
<Similar Trend Time: uptrend1 2025-02-24, 2025-03-05>
<Reason: because similar macro as Bitcoin price rally driven by Trump administration's announcement of a new crypto strategic reserve holding BTC, combined with improved market liquidity and institutional adoption through spot Bitcoin ETFs like iShares Bitcoin Trust, creating positive sentiment around cryptocurrency assets, micro as MicroStrategy's stock rose 4.5% due to its status as the largest corporate Bitcoin holder with approximately 190,000 BTC on its balance sheet, directly benefiting from Bitcoin's price appreciation, with analysts projecting potential doubling of MSTR's value from current $269 levels as Bitcoin premium expands>
<Similar Trend Price: start: 2025-02-24, end: 2025-03-05, day_avg_return: 1.615%, slope: 1.41, max_return: 5.00%, estimate_price: $308.55, duration: 9.0 days, return_variance: 0.008338774444748202, volatility: 9.13%>

<Similar Trend Time: uptr

In [11]:
print(final_results["Macro_Analyst_Agent_Result"])

**OPPORTUNITY**
• **FACT:** Strong economic expansion with robust GDP growth
  - **EVIDENCE:** 0.73% quarter-over-quarter real GDP growth (annualizing to ~3.0%)
  - **RESULT:** Creates favorable conditions for risk assets like cryptocurrency as economic strength supports investor confidence and capital flows into alternative investments, benefiting MSTR's bitcoin-focused strategy

• **FACT:** Significant decline in mortgage rates indicating easing financial conditions
  - **EVIDENCE:** 30-year fixed mortgage rates down -4.1% and 15-year rates down -5.3% over three months
  - **RESULT:** Suggests broader financial market liquidity improvement, which typically supports cryptocurrency valuations and consequently MSTR's bitcoin treasury value

• **FACT:** Stable monetary policy with Federal Reserve on pause
  - **EVIDENCE:** Federal Funds Rate holding steady at 4.33%
  - **RESULT:** Provides predictability for crypto markets, reducing interest rate volatility risk that typically negatively

In [14]:
print(final_results['Revenue_Segmentation_Agent_Result'])

✅ Using fresh data for MSTR. **PART 1: REVENUE SEGMENTATION BREAKDOWN**
• MicroStrategy Analytics Platform (Enterprise Analytics Software): 85% of total revenue
• Mobile Applications and Mobile Software Solutions: 10% of total revenue
• Cloud-based Business Intelligence and Analytics Solutions: 5% of total revenue
• Bitcoin Treasury Operations and Digital Asset Strategy: Not applicable (capital appreciation, not revenue)

**PART 2: IMPACTED BUSINESS LINES**
• Bitcoin Treasury Operations and Digital Asset Strategy: Government stimulus policies can impact Bitcoin's value through monetary policy effects on risk assets and inflation expectations, while cryptocurrency market conditions directly affect the capital appreciation of Bitcoin holdings
• MicroStrategy Analytics Platform (Enterprise Analytics Software): Government stimulus policies may increase enterprise IT spending and data analytics demand as organizations seek to optimize operations during economic changes
• Cloud-based Busines